# 01 — RFM Customer Segmentation

Goal: build a plausible customer base and segment it by recency, frequency and monetary value (RFM). Data is synthetic and created for learning.

## 1. Create realistic transaction data
Each customer has a skewed purchase frequency and spend, typical of retail datasets.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
n_customers=1500
customer_ids=np.arange(10001,10001+n_customers)
# Heavy-tailed purchasing: most clients make few orders; a minority are frequent
orders_per_customer=np.random.choice([1,2,3,4,5,6,8,10], n_customers, p=[.34,.24,.16,.10,.07,.04,.03,.02])
rows=[]
for cid, n in zip(customer_ids,orders_per_customer):
    dates=pd.to_datetime('2025-01-01')+pd.to_timedelta(np.random.randint(0,365,size=n),unit='D')
    amounts=np.random.lognormal(mean=3.8,sigma=.65,size=n).round(2)
    rows.extend((cid,d,a) for d,a in zip(dates,amounts))
orders=pd.DataFrame(rows,columns=['customer_id','order_date','amount'])
orders.head()

## 2. Calculate R, F, M and score clients
R is reverse-ranked so recent purchasers receive higher scores. Frequency uses fixed thresholds (1, 2–3, 4+ purchases) rather than quantiles.

In [ ]:
snapshot=pd.Timestamp('2026-01-01')
rfm=orders.groupby('customer_id').agg(last_order=('order_date','max'),frequency=('order_date','count'),monetary=('amount','sum')).reset_index()
rfm['recency_days']=(snapshot-rfm.last_order).dt.days
rfm['R']=pd.qcut(rfm.recency_days,5,labels=[5,4,3,2,1]).astype(int)
rfm['F']=np.select([rfm.frequency>=4,rfm.frequency>=2],[5,3],default=1)
rfm['M']=pd.qcut(rfm.monetary,5,labels=[1,2,3,4,5]).astype(int)
def label(row):
 r,f=row.R,row.F
 if r>=4 and f>=4:return 'Champions'
 if r>=3 and f>=3:return 'Loyal'
 if r>=4 and f<=2:return 'New / recent'
 if r<=2 and f>=3:return 'At risk'
 return 'Sleeping'
rfm['segment']=rfm.apply(label,axis=1)
rfm.head()

## 3. Segment overview

In [ ]:
segment_counts=rfm.segment.value_counts().sort_values(ascending=False)
print(segment_counts)
ax=segment_counts.plot(kind='bar',figsize=(9,4),title='Customers by RFM segment',ylabel='Customers',xlabel='Segment',rot=25)
plt.tight_layout(); plt.show()

## 4. Monetary value by segment

In [ ]:
seg_summary=rfm.groupby('segment').agg(customers=('customer_id','count'),avg_orders=('frequency','mean'),avg_spend=('monetary','mean'),total_spend=('monetary','sum')).sort_values('total_spend',ascending=False)
seg_summary.round(2)
seg_summary['total_spend'].plot(kind='bar',figsize=(9,4),title='Total customer spend by segment',ylabel='Spend (currency units)')
plt.tight_layout(); plt.show()

## 5. Business takeaway
Interpret the charts: prioritize high-value Champions for retention and advocacy; use a measured reactivation test for At Risk clients; use low-cost onboarding and reactivation for New/Sleeping groups. Avoid assuming that every high-spend client needs a discount.